In [14]:
from huggingface_hub import snapshot_download

# Download model to a specific location\n",
model_path = snapshot_download(
    repo_id="nlptown/bert-base-multilingual-uncased-sentiment",
    cache_dir="./tmp/models"
)

# Then load from local path
from transformers import pipeline
classifier = pipeline(
    "sentiment-analysis",
    model=model_path,
    # local_files_only=True
)

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 28093.13it/s]
The tokenizer you are loading from './tmp/models/models--nlptown--bert-base-multilingual-uncased-sentiment/snapshots/8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


In [15]:
# Usage
result = classifier("This product is amazing!")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

[{'label': '5 stars', 'score': 0.8754845857620239}]


In [32]:
model_path = snapshot_download(
    repo_id="cardiffnlp/twitter-roberta-base-sentiment",
    cache_dir="../../final_pipeline/models/overall_sentiment"
)

# Option 2: 3-class (positive/negative/neutral)
classifier = pipeline(
    "sentiment-analysis",
    model=model_path,
)

Fetching 10 files: 100%|██████████| 10/10 [00:19<00:00,  1.98s/it]
Device set to use cuda:0


In [26]:
# Usage
result = classifier("This product is amazing!")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

result = classifier("yesterday I bought an apple")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

result = classifier("This product is terrible!")
print(result)  # [{'label': '5 stars', 'score': 0.85}]

[{'label': 'LABEL_2', 'score': 0.9905126690864563}]
[{'label': 'LABEL_1', 'score': 0.5134272575378418}]
[{'label': 'LABEL_0', 'score': 0.9816111326217651}]


In [16]:
# # Option 1: Binary (positive/negative)
# classifier = pipeline(
#     "sentiment-analysis",
#     model="distilbert-base-uncased-finetuned-sst-2-english"
# )

# # Option 2: 3-class (positive/negative/neutral)
# classifier = pipeline(
#     "sentiment-analysis",
#     model="cardiffnlp/twitter-roberta-base-sentiment"
# )

In [17]:
import pandas as pd

df_1 = pd.read_csv("../../data/cleaned_data/cleaned_Study_1_reviews.csv")

df_1.columns

Index(['ID', 'finalReview', 'Satisfaction_final', 'cleaning_service_quality',
       'order_packaging', 'communication_and_responsiveness',
       'Driver_professionalism', 'Service_speed',
       'cleaning_service_quality_sentiment', 'order_packaging_sentiment',
       'communication_and_responsiveness_sentiment',
       'Driver_professionalism_sentiment', 'Service_speed_sentiment',
       'Overall_review_sentiment', 'Emotional_intensity_LLM', 'lang'],
      dtype='object')

In [33]:
df_1["finalReview"] = df_1["finalReview"].fillna("").astype(str)

texts = df_1["finalReview"].tolist()

final_label = {'LABEL_0': 'Negative', 'LABEL_1': 'Neutral', 'LABEL_2': 'Positive'}

results = classifier(texts, batch_size=32)
df_1["Overall_review_sentiment"] = [final_label[r["label"]] for r in results]


In [34]:
df_1.head()

,ID,finalReview,Satisfaction_final,cleaning_service_quality,order_packaging,communication_and_responsiveness,Driver_professionalism,Service_speed,cleaning_service_quality_sentiment,order_packaging_sentiment,communication_and_responsiveness_sentiment,Driver_professionalism_sentiment,Service_speed_sentiment,Overall_review_sentiment,Emotional_intensity_LLM,lang
0,1,My order was to dryclean! All suits came back ...,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN,en
1,2,poor experience. jacket not cleaned properly.,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN,en
2,3,The clean laundry came in a bag that had a sme...,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN,en
3,4,not happy with the service. i received multipl...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN,en
4,5,its a very expensive service.,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,NaN,en
